## Обучаем (несколько эпох) ResNet18 на новом датасете

In [ ]:
import sys
sys.path.append('../..')

from pathlib import Path
import os
from glob import glob

import cv2
from PIL import Image
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from src.model.baseline_cnn import LitCNN
from src.model.resnet_18 import LitResNet18
from src.model.dataset import ImageDataset

from tqdm import tqdm
tqdm.pandas()

%load_ext autoreload
%autoreload 2

W0102 18:01:43.819000 74900 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0102 18:01:43.820000 74900 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0102 18:01:43.820000 74900 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0102 18:01:43.820000 74900 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0102 18:01:43.821000 74900 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0102 18:01:43.821000 74900 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0102 18:01:43.822000 74900 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0102 18:01:43.822000 74900 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels


In [4]:
# Изображения хранятся в директории так, что каждой модели кроссовок соответствует
# своя папка. Чтобы можно было рассчитывать статистики, мы собираем в датафрейм относительный
# путь до каждого из изображений, и рассчитываем агрегаты на основе этого датафрейма
from src.data.utils.eda_utils import directory_to_dataframe

path_to_dataset = Path("../../data/01_raw/search-dataset-images")
df = directory_to_dataframe(path_to_dataset)
df

,path,sneaker_class
0,Vans Кеды Upland/236.jpeg,Vans Кеды Upland
1,Vans Кеды Upland/47.jpeg,Vans Кеды Upland
2,Vans Кеды Upland/261.jpeg,Vans Кеды Upland
3,Vans Кеды Upland/148.jpeg,Vans Кеды Upland
4,Vans Кеды Upland/10.jpeg,Vans Кеды Upland
...,...,...
11158,Under Armour Кроссовки UA Phantom 4/130.jpeg,Under Armour Кроссовки UA Phantom 4
11159,Under Armour Кроссовки UA Phantom 4/3.jpeg,Under Armour Кроссовки UA Phantom 4
11160,Under Armour Кроссовки UA Phantom 4/87.jpeg,Under Armour Кроссовки UA Phantom 4
11161,Under Armour Кроссовки UA Phantom 4/167.jpeg,Under Armour Кроссовки UA Phantom 4


In [9]:
def is_corrupted(path):
    try:
        Image.open(path_to_dataset / path)
    except:
        return 1
    return 0
df['corrupted_flg'] = df['path'].progress_apply(is_corrupted)
df['corrupted_flg'].mean()

100%|██████████| 11163/11163 [00:02<00:00, 4069.10it/s]


np.float64(0.0011645614978052494)

In [10]:
df = df[df['corrupted_flg'] == 0]
df

,path,sneaker_class,corrupted_flg
0,Vans Кеды Upland/236.jpeg,Vans Кеды Upland,0
1,Vans Кеды Upland/47.jpeg,Vans Кеды Upland,0
2,Vans Кеды Upland/261.jpeg,Vans Кеды Upland,0
3,Vans Кеды Upland/148.jpeg,Vans Кеды Upland,0
4,Vans Кеды Upland/10.jpeg,Vans Кеды Upland,0
...,...,...,...
11158,Under Armour Кроссовки UA Phantom 4/130.jpeg,Under Armour Кроссовки UA Phantom 4,0
11159,Under Armour Кроссовки UA Phantom 4/3.jpeg,Under Armour Кроссовки UA Phantom 4,0
11160,Under Armour Кроссовки UA Phantom 4/87.jpeg,Under Armour Кроссовки UA Phantom 4,0
11161,Under Armour Кроссовки UA Phantom 4/167.jpeg,Under Armour Кроссовки UA Phantom 4,0


In [11]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["sneaker_class"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.25,
    stratify=train_df["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

,path,sneaker_class,corrupted_flg
3781,Tendance Кроссовки/174.jpeg,Tendance Кроссовки,0
8356,PUMA Кроссовки RS-X Efekt PRM/227.jpeg,PUMA Кроссовки RS-X Efekt PRM,0
7558,Vans Кеды Knu Skool/92.jpeg,Vans Кеды Knu Skool,0
3593,Under Armour Кроссовки UA Charged Rogue 5/136....,Under Armour Кроссовки UA Charged Rogue 5,0
5233,Under Armour Кроссовки UA Charged Surge 4/22.jpeg,Under Armour Кроссовки UA Charged Surge 4,0


(6690, 3)

,path,sneaker_class,corrupted_flg
8318,PUMA Кроссовки RS-X Efekt PRM/142.jpeg,PUMA Кроссовки RS-X Efekt PRM,0
10485,Vans Кеды Old Skool/38.jpeg,Vans Кеды Old Skool,0
5502,X-Plode Кроссовки/8.jpeg,X-Plode Кроссовки,0
7602,Vans Кеды Knu Skool/124.jpeg,Vans Кеды Knu Skool,0
7843,PUMA Кроссовки Hypnotic LS/133.jpeg,PUMA Кроссовки Hypnotic LS,0


(2230, 3)

,path,sneaker_class,corrupted_flg
5139,Under Armour Кроссовки UA Charged Surge 4/165....,Under Armour Кроссовки UA Charged Surge 4,0
9658,Reebok Кеды CLUB C GROUNDS UK/4.jpeg,Reebok Кеды CLUB C GROUNDS UK,0
4695,Kari Кеды/232.jpeg,Kari Кеды,0
8654,Nike Кроссовки Nike Zoom Vomero 5/183.jpeg,Nike Кроссовки Nike Zoom Vomero 5,0
4811,Nike Кроссовки AIR MAX 90/51.jpeg,Nike Кроссовки AIR MAX 90,0


(2230, 3)

In [12]:
train_paths = train_df["path"].tolist()
val_paths   = val_df["path"].tolist()
test_paths  = test_df["path"].tolist()

train_labels = train_df["sneaker_class"].tolist()
val_labels   = val_df["sneaker_class"].tolist()
test_labels  = test_df["sneaker_class"].tolist()

In [13]:
# Аугментация и приведение всех изображений к одному масштабу

train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

In [14]:
train_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=train_paths,
    labels=train_labels,
    augmenter=train_tfms
)

val_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=val_paths,
    labels=val_labels,
    augmenter=val_tfms
)

test_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=test_paths,
    labels=test_labels,
    augmenter=val_tfms
)

In [15]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

In [16]:
resnet_model = LitResNet18(num_classes=df["sneaker_class"].nunique(),
                           lr=0.001)

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=pl.loggers.TensorBoardLogger("lightning_logs", name="resnet18_new_dataset"),
    log_every_n_steps=1
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


In [17]:
import warnings 
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

trainer.fit(resnet_model, train_loader, val_loader)

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ ResNet           │ 11.2 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pyt
ree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


...

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


`Trainer.fit` stopped: `max_epochs=10` reached.


In [18]:
pred_batches = trainer.predict(resnet_model, dataloaders=[test_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred))


/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


              precision    recall  f1-score   support

           0       0.86      0.44      0.58        55
           1       0.54      0.61      0.57        54
           2       0.69      0.74      0.71        54
           3       0.78      0.75      0.76        56
           4       0.66      0.91      0.76        57
           5       0.91      0.78      0.84        54
           6       0.87      0.71      0.78        58
           7       0.71      0.86      0.78        57
           8       0.98      0.73      0.84        56
           9       0.96      0.91      0.94        56
          10       0.64      0.70      0.67        54
          11       0.94      0.59      0.73        54
          12       0.90      0.63      0.74        57
          13       0.38      0.91      0.54        56
          14       0.57      0.77      0.66        52
          15       0.31      0.94      0.47        51
          16       0.68      0.46      0.55        57
          17       0.72    

In [28]:
idx_to_class = {value: key for key, value in test_dataset.class_to_idx.items()}
y_true_class = pd.Series(y_true).map(idx_to_class)
y_pred_class = pd.Series(y_pred).map(idx_to_class)
print(classification_report(y_true_class, y_pred_class))

                                               precision    recall  f1-score   support

                                    Kari Кеды       0.86      0.44      0.58        55
                               Kari Кроссовки       0.54      0.61      0.57        54
                            Maison David Кеды       0.69      0.74      0.71        54
                       Maison David Кроссовки       0.78      0.75      0.76        56
                     Nike Кеды Dunk Low Retro       0.66      0.91      0.76        57
                      Nike Кеды FIELD GENERAL       0.91      0.78      0.84        54
                   Nike Кеды NIKE AIR FORCE 1       0.87      0.71      0.78        58
                    Nike Кроссовки AIR MAX 90       0.71      0.86      0.78        57
              Nike Кроссовки AIR PEGASUS 2005       0.98      0.73      0.84        56
            Nike Кроссовки Nike Zoom Vomero 5       0.96      0.91      0.94        56
                  PUMA Кеды CA Pro Classic